# TP 1 — Du texte brut au premier modèle bigramme

**Objectif** : transformer un corpus en prédictions supervisées, puis entraîner une table qui prédit le caractère suivant.

Plan : 1. corpus → 2. `encode`/`decode` → 3. contextes/cibles → 4. mini-batchs → 5. modèle bigramme → 6-7. générer, entraîner, comparer.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

## Étape 1 — Télécharger et inspecter le corpus

In [2]:
from urllib.request import urlretrieve
import os

url = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
if not os.path.exists("input.txt"):
    urlretrieve(url, "input.txt")
print(os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text))
print(text[:1000])

/root/transformer-tps/notebooks/input.txt
1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this

In [3]:
print("Retours à la ligne :", text.count("\n"))
print("Majuscules :", sum(c.isupper() for c in text), "| minuscules :", sum(c.islower() for c in text))
print("Ponctuation :", sorted({c for c in text if not c.isalnum() and not c.isspace()}))

Retours à la ligne : 40000
Majuscules : 88754 | minuscules : 762324
Ponctuation : ['!', '$', '&', "'", ',', '-', '.', ':', ';', '?']


### Réponses
- **Unité d'un token** : un **caractère** (lettre, espace, ponctuation, `\n`…). Pas de mots ni de sous-mots.
- **`encoding="utf-8"`** : garantit le même décodage des octets quel que soit l'OS (sinon Windows peut utiliser cp1252 et casser certains caractères ou lever une erreur).
- **Déjà séparé en exemples ?** Non : c'est un seul long texte continu (~1,1 M caractères). C'est nous qui découpons ensuite des fenêtres de longueur `block_size`.

## Étape 2 — Passer des caractères aux entiers

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("vocab_size =", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

message = "hii there"
print(encode(message))
print(decode(encode(message)))
assert decode(encode(message)) == message


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size = 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### Réponses
- **Pourquoi trier `chars` ?** `set` n'a pas d'ordre garanti : sans tri, les identifiants changeraient d'une exécution à l'autre et un modèle sauvegardé deviendrait incohérent. Le tri rend le vocabulaire **déterministe et reproductible**.
- **`decode(encode(m)) == m`** garantit que les deux tables sont **parfaitement inverses** (bijection) : aucune information n'est perdue lors de la tokenisation.

## Étape 3 — Fabriquer les prédictions attendues

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(context.tolist(), "->", target.item(), f"  ({decode(context.tolist())!r} -> {itos[target.item()]!r})")

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
[18] -> 47   ('F' -> 'i')
[18, 47] -> 56   ('Fi' -> 'r')
[18, 47, 56] -> 57   ('Fir' -> 's')
[18, 47, 56, 57] -> 58   ('Firs' -> 't')
[18, 47, 56, 57, 58] -> 1   ('First' -> ' ')
[18, 47, 56, 57, 58, 1] -> 15   ('First ' -> 'C')
[18, 47, 56, 57, 58, 1, 15] -> 47   ('First C' -> 'i')
[18, 47, 56, 57, 58, 1, 15, 47] -> 58   ('First Ci' -> 't')


### Réponses
1. Un bloc de `block_size = 8` produit **8 prédictions** (contextes de longueur 1 à 8), à partir de 9 caractères.
2. **Ne pas mélanger train/val** : la validation doit mesurer la généralisation sur du texte jamais vu. Si les deux se recouvrent, la loss de validation est optimiste et on ne détecte plus le surapprentissage. On coupe en bloc (90/10) plutôt qu'aléatoirement pour éviter que des fenêtres voisines (presque identiques) fuient d'un côté à l'autre.
3. **t = 3** : contexte = `x[:4]` = `"Firs"`, cible = `y[3]` = `"t"`.

## Étape 4 — Échantillonner un mini-batch

In [6]:
batch_size = 4
block_size = 8

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


### Réponses
1. **Retirer `block_size` à la borne** : `y` lit jusqu'à l'indice `i + block_size`. Si `i` pouvait aller jusqu'à `len - 1`, on sortirait du tenseur. Avec `i < len - block_size`, le dernier indice lu est `len - 1` ✔.
2. **B = 32** : 32 séquences en parallèle → gradient moins bruité (moyenne sur 256 prédictions au lieu de 32), plus de mémoire, plus de calcul par étape, mais même « vision » du contexte.
3. **T = 64** : chaque séquence est plus longue → plus de prédictions par séquence et surtout un contexte potentiel plus long. Pour le bigramme ça ne change rien (il ne regarde qu'un caractère), mais pour un Transformer ça permet d'exploiter 64 caractères de passé.

## Étape 5 — Une table produit les logits

In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # table (C, C) : ligne = caractère courant, colonnes = scores du suivant
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # idx: (B,T) -> logits: (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: (B,T) contexte courant
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]                    # (B,C) : dernier pas de temps
            probs = F.softmax(logits, dim=-1)            # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B,1)
            idx = torch.cat((idx, idx_next), dim=1)      # (B,T+1)
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)   # (B*T, C) car aplati
print(loss.item(), "vs -ln(1/C) =", torch.log(torch.tensor(vocab_size)).item())

torch.Size([32, 65])
5.036386013031006 vs -ln(1/C) = 4.174387454986572


### Réponses
- **Pourquoi C × C ?** Le modèle bigramme associe à chaque caractère (C lignes) une distribution sur le caractère suivant (C colonnes). La « ligne d'embedding » **est directement** le vecteur de logits.
- **Dans le TP 4**, cette table est **scindée en deux** : `nn.Embedding(vocab_size, n_embd)` (d_emb = 64, l'identité du token) puis `lm_head = nn.Linear(n_embd, vocab_size)` qui ramène la représentation contextualisée vers les C logits.
- La loss initiale vaut ≈ 5,0, alors qu'une prédiction **uniforme** donnerait ln(65) ≈ 4,17. Les poids aléatoires (N(0,1)) produisent une distribution déjà « confiante » mais au hasard : c'est pire qu'uniforme. Un bon point de départ serait donc ≈ 4,17.

## Étapes 6 et 7 — Générer, puis apprendre

In [8]:
def sample(model, n=300):
    idx = torch.zeros((1, 1), dtype=torch.long)  # token 0 = '\n'
    return decode(model.generate(idx, max_new_tokens=n)[0].tolist())

print("=== AVANT entraînement ===")
print(sample(model))

=== AVANT entraînement ===

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheHd:.?nA!LgjgL&jcM-sZW!HFsWt'rv.x;z&hPyd'XHOxnAcj.erRjcLxxVFSrOpW.oeGJ Ai$yi&!mRfeopx,f.e..iBgxVrl-VThV?W?qF;roi!NxbR;zyDlfd&DH'X;PgqTOMcL
HvR;tg;tEkFST$Cil;AkkEptkGClcJ NoPTeZLPX.B-aft'ZEl'rJfEDm&UZ
RE


In [9]:
def train(model, steps, lr=1e-3, every=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if every and step % every == 0:
            print(step, round(loss.item(), 4))
    return loss.item()

torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
print("loss après 100 étapes :", train(model, 100))
print(sample(model))

loss après 100 étapes : 5.016654014587402

prOLTXyGHjNWqUwcz-BBi
?&SAcxlfRF$
l,egKujQ&vfFbOHAuW?KphiWGjbrZSqtRwBPlvUXiT
hirQRTEHnAcnSboiFGJEa!z3kzPHdgkI'MkJ&vWTpErgFGGRsBi.YWDMAW!UEXlkUZpZboc;ZIu!SSPdL,T' AddZJBUvf Bjx;blxl$GXKw'fRSGSliPSPZv!m3Q'XEJzPKuY,iN&Ai
ErAE!U;AdN&LNCKb3sr.RdlxBhqPmGSfiEV;hOmxhD.iNGMbl-QbWIMU'aSXcqAWWi!o'wBkb .xIBHYw:


In [10]:
batch_size = 32  # batch plus grand pour un gradient moins bruité
torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
print("loss après 1 000 étapes :", train(model, 1000, lr=1e-2, every=200))
print(sample(model, 500))

0 4.6485


200 3.1947


400 2.6953
600 2.57


800 2.3996


loss après 1 000 étapes : 2.4111454486846924

S:
Land b'Mckesc; de ndg LARDHENGo wleRTINongefe Ise lort tare!
JAMENTHOYout thive.
Innod llar y call.
Bjeif wst isechewgbj$dingr, the, wes g'siveranem s he th ie regoiFRIKI CAA tinth as wenthon st;

PAnoof therothare, 'sur t tyowite th whean qurd?O:
ngho apaknpuplorGUn acLot llo lkes be hy Card te n &QQUCE t,
S:
INNGLIVsthouthen nClg,

Fous
MOqurVo!
Befony hinmaprst ak!

hal burg ancis, $E:
Yo bes hathaveande theY:
Pre fatanthr-usth cathad
R:
He t,
M:
Bu aioused tallofa t roufid or.
tas h manou


In [11]:
# Bonus : jusqu'à convergence (~2,45 = plancher d'un bigramme)
print("loss après 10 000 étapes :", train(model, 10000, lr=1e-2, every=2000))
print(sample(model, 500))

0 2.46


2000 2.438


4000 2.5487


6000 2.4953


8000 2.5261


loss après 10 000 étapes : 2.3372979164123535


I s s not
CAne ligons,
My ye houncot wiohes,
Thint ss hour.
Are Inde avite alisety; ay AD:

LI l I'd y ty
I'loulow scow west k'ldstate ranard p re hearid ng.
I t, mas ownlemath a thiner.

CO:
GAngut OF teeor:

I angout, frure?
Wouswis at g sarulewild and, al LYowamakn meatan alo mistlouf an Y:
te hathan nghiereroust I a fef alfeichak!
IZougisedotorend faindsimothimes he igrm what
Andr:
O:

Aseid ouget l ose, wndid mesemy y:

Tos le ou houso, thesuptho ake m i, ARI:
DWet s scken ffouneinongury e


### Bilan du TP 1
- **Dernier pas de temps en génération** : le modèle produit des logits pour toutes les positions, mais seule la dernière prédit le **prochain** caractère encore inconnu ; les autres prédisent des caractères qu'on connaît déjà.
- **Aplatir B × T pour la loss** : `F.cross_entropy` attend `(N, C)` et `(N,)`. Chaque position de chaque séquence est une classification indépendante ; on en fait donc N = B·T exemples.
- **Ce que l'entraînement modifie** : uniquement les C² = 4 225 coefficients de la table. Chaque ligne apprend les fréquences « caractère suivant | caractère courant ».
- **Avant / après** : la loss part de ~5,0 et reste à ~5,0 après 100 étapes (lr 1e-3, B=4 : ça bouge à peine, texte toujours aléatoire). Avec 1 000 étapes (B=32, lr 1e-2) elle tombe à ~2,4, et plafonne autour de ~2,45 ensuite : c'est le meilleur qu'un bigramme puisse faire. Le texte passe d'un charabia uniforme à quelque chose qui a des **espaces, des retours à la ligne, des majuscules après `\n`, des syllabes plausibles** — mais aucun mot cohérent, car le modèle ne voit qu'**un seul** caractère de contexte. C'est la limite que l'attention va lever.